In [1]:
%%capture
!pip install --upgrade transformers bitsandbytes accelerate peft datasets wandb contractions spacy tqdm

In [2]:
import torch
import wandb
import pandas as pd
from transformers import AutoModelForSequenceClassification, AutoTokenizer, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model, TaskType
from datasets import Dataset, DatasetDict
import numpy as np
import re
from sklearn.metrics import precision_score, f1_score, recall_score, accuracy_score, balanced_accuracy_score
from sklearn.model_selection import train_test_split
from contractions import fix
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from tqdm import tqdm
from tqdm.contrib.concurrent import process_map
import random

# Login to Weights & Biases (W&B) - Required in Kaggle
wandb.login(key="REDACTED")  # Replace with your actual API key
wandb.init(project="Spring2025", reinit=True)


wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: dhruvjoshi892000 (dhruvjoshi892000-pace-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


In [3]:
# Ensure GPU availability
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using {device} for training")

Using cuda for training


In [4]:
# 🔧 Set Random Seed for Reproducibility
random.seed(42)
np.random.seed(42)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

import os
os.environ['CUDA_LAUNCH_BLOCKING'] = '1'


In [11]:
import spacy

# Load spaCy model with GPU support
spacy.require_gpu()  # Ensure spaCy runs on GPU
nlp = spacy.load("en_core_web_sm")

# Verify if GPU is used
print("spaCy is using GPU:", spacy.prefer_gpu())


# Load spaCy's English NLP model
nlp = spacy.load("en_core_web_sm")

def preprocess_text(text):
    """
    Preprocess legal text for analysis using spaCy.
    Steps:
    - Lowercasing
    - Removing HTML tags
    - Removing URLs
    - Expanding contractions
    - Removing special characters
    - Removing numbers
    - Removing stop words
    - Lemmatization using spaCy
    - Removing extra whitespace
    """
    # Lowercase the text
    text = text.lower()

    # Remove HTML tags and URLs
    text = re.sub(r'<.*?>', '', text)
    text = re.sub(r'http\S+|www\.\S+', '', text)

    # Expand contractions (optional)
    try:
        text = fix(text)
    except ImportError:
        pass  # Skip if the 'contractions' package is not available

    # Remove special characters and numbers (excluding spaces)
    text = re.sub(r'[^a-z\s]', '', text)

    # Process text using spaCy
    doc = nlp(text)

    # Lemmatize and remove stopwords
    text = ' '.join([token.lemma_ for token in doc if not token.is_stop])

    # Remove extra whitespace
    text = ' '.join(text.split())

    return text


# Define tokenization function
def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=512)

# Load dataset from CSV
df = pd.read_csv("/content/279_labels_data.csv")  # Adjust file path if needed

# Apply preprocessing
# df = df[:10]
# df["text"] = df["text"].apply(preprocess_text)
processed_texts = [preprocess_text(text) for text in tqdm(df['text'], desc="Preprocessing texts")]

# Convert DataFrame to Hugging Face Dataset
dataset = Dataset.from_pandas(df)

# Split dataset into train and validation
dataset = dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = dataset['train']
eval_dataset = dataset['test']


# Load tokenizer
model_name = "nlpaueb/legal-bert-base-uncased"  # Modify as needed
tokenizer = AutoTokenizer.from_pretrained(model_name)


# Apply tokenization
train_dataset = train_dataset.map(tokenize_function, batched=True)
eval_dataset = eval_dataset.map(tokenize_function, batched=True)



spaCy is using GPU: True


Preprocessing texts: 100%|██████████| 8417/8417 [41:14<00:00,  3.40it/s]


Map:   0%|          | 0/7575 [00:00<?, ? examples/s]

Map:   0%|          | 0/842 [00:00<?, ? examples/s]

In [12]:
# Set format for PyTorch
train_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
eval_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])

In [13]:
# Load pre-trained model
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=279).to(device)

# Adjusted LoRA Configuration for Long-Form Text Classification
lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=16,
    lora_alpha=32,
    lora_dropout=0.4,
    target_modules=[
        "attention.self.query",
        "attention.self.key",
        "attention.self.value",
        "attention.output.dense",
        "intermediate.dense",
        "output.dense"
    ],
    bias="none",
    inference_mode=False
)

# Integrate LoRA with the model
model = get_peft_model(model, lora_config).to(device)
model.print_trainable_parameters()


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at nlpaueb/legal-bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


trainable params: 2,868,759 || all params: 112,565,550 || trainable%: 2.5485


In [14]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=1)
    accuracy = accuracy_score(labels, predictions)
    balanced_acc = balanced_accuracy_score(labels, predictions)
    precision = precision_score(labels, predictions, average='weighted', zero_division=0)
    recall = recall_score(labels, predictions, average='weighted', zero_division=0)
    f1 = f1_score(labels, predictions, average='weighted', zero_division=0)
    return {
        "accuracy": accuracy,
        "balanced_accuracy": balanced_acc,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }


In [15]:
# training_args = TrainingArguments(
#     output_dir="/kaggle/working/results",
#     evaluation_strategy="epoch",
#     save_strategy="epoch",
#     logging_dir="/kaggle/working/logs",
#     label_names=["labels"],
#     logging_steps=10,
#     num_train_epochs=20,  # More epochs for better legal text learning
#     per_device_train_batch_size=4,  # Lower batch size for stability
#     per_device_eval_batch_size=4,
#     learning_rate=2e-5,  # Reduce LR for better fine-tuning
#     warmup_ratio=0.1,  # Stabilize early training
#     weight_decay=0.01,
#     metric_for_best_model="balanced_accuracy",
#     load_best_model_at_end=True,
#     report_to=["wandb"],  # Enable W&B logging
#     fp16=torch.cuda.is_available(),  # Use FP16 if GPU is available
# )

training_args = TrainingArguments(
    output_dir="./results",
    evaluation_strategy="epoch",  # Evaluate after every epoch
    save_strategy="epoch",        # Save the model every epoch
    learning_rate=2e-5,           # Reduced LR for stability
    per_device_train_batch_size=16,  # Increase if memory allows
    per_device_eval_batch_size=16,
    num_train_epochs=20,         # Keep training epochs
    weight_decay=0.05,           # Regularization to prevent overfitting
    logging_dir="./logs",
    logging_steps=50,            # Log progress every 50 steps
    load_best_model_at_end=True, # Load the best checkpoint
    metric_for_best_model="eval_loss",  # Select the best model based on validation loss
    lr_scheduler_type="cosine",  # Cosine decay learning rate
    warmup_ratio=0.1,            # 10% warmup steps
    gradient_accumulation_steps=2,  # Stabilizes training for smaller batch sizes
    fp16=True,                   # Mixed precision training
    report_to="wandb",           # Log results to Weights & Biases
)

/usr/local/lib/python3.11/dist-packages/transformers/training_args.py:1611: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


In [16]:
# Initialize Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    compute_metrics=compute_metrics
)

# Start training
trainer.train()

# Evaluate the model
results = trainer.evaluate()
print(results)


No label_names provided for model class `PeftModelForSequenceClassification`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


Epoch,Training Loss,Validation Loss,Accuracy,Balanced Accuracy,Precision,Recall,F1
1,5.563800,5.466166,0.014252,0.005025,0.000212,0.014252,0.000418
2,5.162900,5.107763,0.045131,0.010070,0.006200,0.045131,0.009174
3,4.997700,4.959741,0.072447,0.016245,0.011418,0.072447,0.018111
4,4.843900,4.733747,0.109264,0.024040,0.035185,0.109264,0.033467
5,4.587700,4.487051,0.131829,0.031948,0.050079,0.131829,0.049620
6,4.336300,4.279020,0.171021,0.047669,0.090894,0.171021,0.077706
7,4.142100,4.084529,0.209026,0.063889,0.088993,0.209026,0.105632
8,3.983100,3.919355,0.231591,0.073667,0.112213,0.231591,0.126523
9,3.858300,3.784348,0.275534,0.094344,0.126563,0.275534,0.161393
10,3.712200,3.671666,0.286223,0.103741,0.155876,0.286223,0.172207


/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


{'eval_loss': 3.3197412490844727, 'eval_accuracy': 0.3657957244655582, 'eval_balanced_accuracy': 0.13900542414719505, 'eval_precision': 0.20121187443028105, 'eval_recall': 0.3657957244655582, 'eval_f1': 0.2465343952872452, 'eval_runtime': 10.9644, 'eval_samples_per_second': 76.794, 'eval_steps_per_second': 4.834, 'epoch': 20.0}
